# 📈 Forex_DNN Historical Simulation & Backtesting Workbench

This interactive notebook is designed to run historical simulations (backtests) for `MMStrategy`, `SMStrategy` (Stubborn Man), and `UniTStrategy` in the Forex_DNN framework.

### Key Features Included:
1. **Execution Modes**:
   - `DEBUG`: Runs a super fast backtest over the last 500 candles to check compilation and rule execution in seconds.
   - `FULL`: Runs a comprehensive backtest over the full historical data range.
2. **ML Usage Toggle (`ML_USAGE`)**:
   - `True`: Activates the centralized `MLDecisionEngine` with real production model filtering to confirm setups.
   - `False`: Runs purely deterministic, rule-based trading (with fallback structural trend regimes), demonstrating strategy autonomy.
3. **Visual Reports**: Plots the equity curve over time and drawdown profiles.

In [ ]:
import os
import sys

# Ensure we can import from the framework root
framework_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if framework_root not in sys.path:
    sys.path.append(framework_root)
print(f"Framework root added to sys.path: {framework_root}")

In [ ]:
import logging
import shutil
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timezone

from Configs.path_manager import PathManager
from Simulation.simulation_clock import SimulationClock
from Simulation.simulation_account import SimulationAccount
from Simulation.simulation_broker import SimulationBroker
from Simulation.simulation_environment import env
from Simulation.historical_data_feed import HistoricalDataFeed

from Collecting_Data.trading_journal import TradingJournal
from Trade_Execution.position_manager import PositionManager
from Trade_Execution.position_tracker import PositionTracker
from Trade_Execution.drawdown import DrawdownManager
from Trade_Execution.risk_sizing import PositionSizer
from Trade_Execution.exit_manager import ExitManager
from Trade_Execution.send_order import SendOrder

from Strategies.mm_strategy import MMStrategy
from Strategies.sm_strategy import SMStrategy
from ML.ml_decision_engine import MLDecisionEngine

## ⚙️ 1. PARAMETERS & INPUT CONFIGURATION

In [ ]:
# Choose "DEBUG" or "FULL"
MODE = "DEBUG" 

# Toggle machine learning decision engine filtering
ML_USAGE = False

# Strategy Selection: "MM" or "SM"
STRATEGY = "SM"

SYMBOL = "EURUSD"
TIMEFRAME = "M5"
INITIAL_BALANCE = 10000.0
LEVERAGE = 100
JOURNAL_ROOT = "Backtest_Journals_Notebook"

## 🚀 2. BACKTEST ENGINE INITIALIZATION

In [ ]:
PathManager.ensure_all_dirs()

# 1. Resolve raw input file
historical_dir = PathManager.get_relative_path("historical_data")
parquet_path = os.path.join(historical_dir, SYMBOL, f"{TIMEFRAME}.parquet")

if not os.path.exists(parquet_path):
    print(f"\033[91m\033[1m[WARNING] No historical data file found at '{parquet_path}'!\033[0m")
    print("Please run the Data Collection or Processing notebooks first to seed historical data.")
else:
    # 2. Slice if DEBUG mode is enabled
    if MODE == "DEBUG":
        df_full = pd.read_parquet(parquet_path)
        df_slice = df_full.iloc[-1000:].copy() # We slice to 1000 bars for indicators and calculations
        temp_slice_path = os.path.join(PathManager.get_path("temporary"), f"{SYMBOL}_{TIMEFRAME}_slice.parquet")
        df_slice.to_parquet(temp_slice_path, index=False)
        data_files = {(SYMBOL, TIMEFRAME): temp_slice_path}
        print(f"DEBUG mode: sliced dataset to last 1000 candles ({temp_slice_path})")
    else:
        data_files = {(SYMBOL, TIMEFRAME): parquet_path}
        print(f"FULL mode: loaded complete historical file ({parquet_path})")
        
    # 3. Load historical data feed
    data_feed = HistoricalDataFeed()
    for (s, tf), path in data_files.items():
        data_feed.load_csv(s, tf, path)

    timeline = data_feed.get_global_timeline()
    start_time = timeline[0]
    
    # 4. Initialize clock, account and broker simulation
    clock = SimulationClock(start_time)
    account = SimulationAccount(INITIAL_BALANCE, LEVERAGE)
    broker = SimulationBroker(account, clock)
    
    info = {
        "digits": 5, "point": 0.00001, "volume_min": 0.01, "volume_step": 0.01,
        "volume_max": 100.0, "trade_contract_size": 100000, "trade_stops_level": 0,
        "trade_tick_value": 1.0, "trade_tick_size": 0.00001
    }
    broker.set_symbol_info(SYMBOL, info)
    env.set_backtest_mode(broker, clock, account)
    
    # Seek to start
    data_feed.seek_to_time(start_time)
    broker.update_market_price(SYMBOL, df_slice.iloc[0]['Close'] if MODE=="DEBUG" else df_full.iloc[0]['Close'], df_slice.iloc[0]['Close'] if MODE=="DEBUG" else df_full.iloc[0]['Close'])

    # 5. Clean stale state and initialize tracking components
    state_dir = os.path.join(JOURNAL_ROOT, "State")
    shutil.rmtree(state_dir, ignore_errors=True)
    os.makedirs(state_dir, exist_ok=True)
    
    tj = TradingJournal(journal_root=JOURNAL_ROOT, mode="backtest")
    pm = PositionManager(magic_unity=100001, magic_mm=100002)
    pt = PositionTracker(magic_numbers=[100001, 100002], poll_interval_seconds=0, state_file=os.path.join(state_dir, "position_tracker_state.json"))
    dm = DrawdownManager(initial_balance=INITIAL_BALANCE, position_tracker=pt, state_file=os.path.join(state_dir, "drawdown_manager_state.json"))
    ps = PositionSizer()
    em = ExitManager(position_tracker=pt, position_manager=pm, trading_journal=tj, state_file=os.path.join(state_dir, "exit_manager_state.json"))
    so = SendOrder(pm, pt, dm, ps, em, tj, state_file=os.path.join(state_dir, "send_order_state.json"))
    
    # 6. ML Decision Engine
    decision_engine = MLDecisionEngine() if ML_USAGE else None
    
    # 7. Initialize Selected Strategy with ML overrides
    if STRATEGY == "SM":
        # For SMStrategy, we turn shadow_mode=False and ml_filtering=True when ML is active
        strategy = SMStrategy(
            data_feed=data_feed,
            send_order=so,
            trading_journal=tj,
            drawdown_manager=dm,
            symbols=[SYMBOL],
            poll_interval_seconds=0,
            state_file=os.path.join(state_dir, "sm_strategy_state.json"),
            decision_engine=decision_engine,
            shadow_mode=not ML_USAGE,     # Active trading when ML is used, otherwise shadow
            ml_filtering=ML_USAGE
        )
    else:
        strategy = MMStrategy(
            data_feed=data_feed,
            send_order=so,
            trading_journal=tj,
            drawdown_manager=dm,
            symbols=[SYMBOL],
            poll_interval_seconds=0,
            state_file=os.path.join(state_dir, "mm_strategy_state.json"),
            decision_engine=decision_engine,
            shadow_mode=not ML_USAGE,
            ml_filtering=ML_USAGE
        )
        
    print(f"\n[INFO] Backtest system successfully loaded with strategy: '{STRATEGY}' (ML_USAGE={ML_USAGE})")

## 🏎️ 3. BACKTEST SIMULATION EXECUTION

In [ ]:
if 'timeline' in locals():
    print(f"Running historical simulation over {len(timeline)} bars...")
    t0 = time.time()
    
    # For the notebook run, let's also force active trading (shadow_mode=False) so we can see orders being placed!
    strategy.shadow_mode = False
    strategy.ml_filtering = ML_USAGE
    
    # Core loop
    for idx, current_time in enumerate(timeline):
        clock.set_time(current_time)
        data_feed.seek_to_time(current_time)
        
        # Update prices
        bar = data_feed.get_current_bar(SYMBOL, TIMEFRAME)
        if bar is not None:
            broker.update_market_price(SYMBOL, bar['Close'], bar['Close'])
            
        # Poll simulation components sequentially
        pt._poll_cycle()
        em._poll_cycle()
        dm.check()
        strategy._poll_cycle()
        
    print(f"Simulation finished in {time.time() - t0:.2f} seconds.")
    print(f"Final Account Balance: ${account.balance:.2f} (Equity: ${account.equity:.2f})")

## 📊 4. SIMULATION PERFORMANCE DIAGNOSTICS

In [ ]:
# Load positions and reconstruct balance curve
from trade_auditor import TradeAuditor

env.set_backtest_mode(broker, clock, account)
auditor = TradeAuditor(journal_root=JOURNAL_ROOT, mode="backtest")
lifecycles = auditor.reconstruct_all()

print(f"Total Trades Closed: {len(lifecycles)}")

# Build simple balance curve
balance_history = [INITIAL_BALANCE]
trade_times = [start_time]

current_bal = INITIAL_BALANCE
for lf in lifecycles:
    pnl = lf.get("pnl", 0.0)
    current_bal += pnl
    balance_history.append(current_bal)
    trade_times.append(pd.to_datetime(lf.get("close_time", clock.current_time)))
    
plt.figure(figsize=(15, 6))
plt.plot(trade_times, balance_history, marker="o", color="green", linewidth=2, label="Account Balance")
plt.axhline(y=INITIAL_BALANCE, color="red", linestyle=":", label="Initial Deposit")
plt.title(f"[{STRATEGY} on {SYMBOL} {TIMEFRAME}] Balance Equity Growth Curve (ML_USAGE={ML_USAGE})")
plt.xlabel("Simulation Timeline")
plt.ylabel("Balance ($)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()